In [1]:
import os

from google.cloud import vision
from google.cloud.vision_v1 import types

import pandas as pd
import cv2
import numpy as np
import json

import imutils
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

from Read import Download

#Load index list
Year='1942'
Showa='17'


In [2]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'

In [3]:
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [4]:
import argparse
from enum import Enum
import io

from google.cloud import vision
from PIL import Image, ImageDraw
# [END vision_document_text_tutorial_imports]


class FeatureType(Enum):
    PAGE = 1
    BLOCK = 2
    PARA = 3
    WORD = 4
    SYMBOL = 5


def draw_boxes(image, bounds, color):
    """Draw a border around the image using the hints in the vector list."""
    draw = ImageDraw.Draw(image)

    for bound in bounds:
        draw.polygon(
            [
                bound.vertices[0].x,
                bound.vertices[0].y,
                bound.vertices[1].x,
                bound.vertices[1].y,
                bound.vertices[2].x,
                bound.vertices[2].y,
                bound.vertices[3].x,
                bound.vertices[3].y,
            ],
            None,
            color,
        )
    return image



In [5]:
from google.cloud import vision
import cv2
import numpy as np
import matplotlib.pyplot as plt

def get_bounds(document, feature):
    bounds = {}
    bounds['blocks'] = []
    bounds['paragraphs'] = []
    bounds['words'] = []
    
    for page in document.pages:
        for block in page.blocks:
            for paragraph in block.paragraphs:
                for word in paragraph.words:
                    bounds['words'].append(word.bounding_box)
                bounds['paragraphs'].append(paragraph.bounding_box)
            bounds['blocks'].append(block.bounding_box)

    return bounds

def draw_bounds(image, bounds, color, thickness):
    for bound in bounds:
        pts = np.array([(vertex.x, vertex.y) for vertex in bound.vertices])
        xmin, xmax = min(pts[:, 0]), max(pts[:, 0])
        ymin, ymax = min(pts[:, 1]), max(pts[:, 1])
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), color, thickness)

def convert_image(image):
    _, encoded_image = cv2.imencode('.png', image)

    api_image = vision.Image(content=encoded_image.tobytes())

    response = client.document_text_detection(image=api_image)
    document = response.full_text_annotation
    return(document)

In [6]:
def cal_size(Block):
    pts = np.array([(vertex.x, vertex.y) for vertex in Block.bounding_box.vertices])
    xmin, xmax = min(pts[:, 0]), max(pts[:, 0])
    ymin, ymax = min(pts[:, 1]), max(pts[:, 1])
    area=(xmax-xmin)*(ymax-ymin)
    return(area)

def Lowbar(Block):
    pts = np.array([(vertex.x, vertex.y) for vertex in Block.vertices])
    ymin, ymax = min(pts[:, 1]), max(pts[:, 1])
    return(ymax)

def Topbar(Block):
    pts = np.array([(vertex.x, vertex.y) for vertex in Block.vertices])
    ymin, ymax = min(pts[:, 1]), max(pts[:, 1])
    return(ymin)


In [7]:
#Load Data Frame
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+str(Year)+"\\DataFrame_PositionFin.json"
with open(path, 'r') as j:
     dta = json.loads(j.read())

df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+str(Showa)+'.csv')
df=df.drop(df.columns[0], axis=1)

filepath="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year

In [83]:
n=5
Row=df.iloc[n]
Dept,Office=Row['Dept'],Row['Office']

Posi=list(dta[Year][Dept][Office]['Positions'].keys())[1]
display(Posi)

'Clerk1'

In [84]:
Part=2 #Part=int(dta[Year][Dept][Office]['Positions'][Posi])
StrPage=int(dta[Year][Dept][Office]['Positions'][Posi]['StrPage'])
EndPage=int(dta[Year][Dept][Office]['Positions'][Posi]['EndPage'])
PageRange=range(StrPage,EndPage+1)

image=Download(filepath,Dept,Office,Posi,PageRange[0])

In [85]:
document=convert_image(image)
bounds = get_bounds(document, FeatureType.PARA)

Page=document.pages
Blocks=Page[0].blocks

sizeList=[cal_size(Block) for Block in Blocks]
IndexList = [sizeList.index(i) for i in sorted(sizeList, reverse=True)][:Part]
NameBoxList=[Blocks[i].bounding_box for i in IndexList]
NameBoxList=[list((Topbar(Block),Lowbar(Block))) for Block in NameBoxList]


In [86]:
copy=image.copy()
WW=copy.shape[:2][1]
for part in NameBoxList:    
    TopLine=min(part)
    BtmLine=max(part)

    cv2.line(copy,(0,TopLine-10),(WW,TopLine-10),(225,0,0),2)
    cv2.line(copy,(0,BtmLine+10),(WW,BtmLine+10),(225,0,0),2)
    cv2.imshow('',copy)
    cv2.waitKey(0)

In [87]:
copy=image.copy()

TopLine=NameBoxList[0][0]
BtmLine=NameBoxList[0][1]

copy=copy[TopLine-10:BtmLine+10,0:WW]
cv2.imshow('',copy)
cv2.waitKey(0)

-1

In [72]:
success,encoded_image = cv2.imencode('.jpg', copy)
content2 = encoded_image.tobytes()
image_cv2 = types.Image(content=content2)
response =  client.text_detection(image=image_cv2,
                                 image_context={"language_hints": ["zh"]})
texts = response.text_annotations

In [81]:
words=np.array([Word for Word in texts[0].description]).tolist()
words=''.join(words).split("\n")
size=len(max(words, key=len))
Candidates=[d for d in words if len(d)==size-1]
LastLetters=Candidates[-1]

In [82]:
LastLetters

'子難人郎雄遗郎郎銀島'